In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
## Step-1 Load pdf file
loader = PyPDFLoader('attention.pdf')
documents = loader.load()

## Step-2 split the docs 
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
docs = text_splitter.split_documents(documents)

## Step-3 Embeddings 
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

## Step-4 Vectorstores
vector_store = FAISS.from_documents(docs,embeddings)
retriever_obj = vector_store.as_retriever()

## Step-5 LLM object
llm_obj = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

##  Step 6: Prompt with history
prompt = ChatPromptTemplate.from_template("""
Use the following context and chat history to answer the question.

Chat history:
{history}

Context:
{context}

Question:
{input}
""")

## Step 7: Build stuff + retrieval chain
qa_chain = create_stuff_documents_chain(llm_obj,prompt)
rag_chain = create_retrieval_chain(retriever_obj,qa_chain)

## Step 8: Add RunnableWithMessageHistory 
stores = {}
def get_session_history(session_id: str) ->BaseChatMessageHistory:
    if session_id not in stores:
        stores[session_id] = ChatMessageHistory()
    return stores[session_id]

rag_with_history = RunnableWithMessageHistory(rag_chain,get_session_history,input_messages_key='input',history_messages_key='history')


## Step 9: Create session ID and invoke rag with chatHistory

session_id = "session-1"
query1 = "What is attention in transformer?"

response1 = rag_with_history.invoke({"input":query1},config={"configurable":{"session_id":session_id}})

print("response-1:",response1)
print('')
query2 = "Explain again in shortly"
response2 = rag_with_history.invoke({"input":query2},config={"configurable":{"session_id":session_id}})

print("response-2:",response2)

In [ ]:
query3="What did i ask previously?"
response3 = rag_with_history.invoke({"input":query3},config={"configurable":{"session_id":session_id}})

print("response-3:",response3)